# **Import of Libraries needed**

In [1]:
import heapq
import time
import random
import statistics

# **The Dijkstra algorithm we will use**

In [2]:
# Used for recommending a shortest path to the player
# Should give the ordered locations

def dijkstra(graph, start, destination):

    # Check if the locations exist
    if start not in graph or destination not in graph:
        raise ValueError("Invalid starting or destination location")

    # Check that all edges have valid destinations and nonnegative costs
    for node in graph:
        for neighbor, cost in graph[node]:
            if neighbor not in graph:
                raise ValueError("Edge points to an unknown location")
            if cost < 0:
                raise ValueError("Negative route costs are not allowed")

    distances = {node: float('inf') for node in graph}
    previous = {node: None for node in graph}

    distances[start] = 0
    pq = [(0, start)]

    while pq:
        current_dist, current_node = heapq.heappop(pq)

        if current_dist > distances[current_node]:
            continue

        # Destination reached
        if current_node == destination:
            break

        for neighbor, weight in graph[current_node]:
            distance = current_dist + weight

            if distance < distances[neighbor]:
                distances[neighbor] = distance
                previous[neighbor] = current_node

                heapq.heappush(pq, (distance, neighbor))

    # No route exists
    if distances[destination] == float('inf'):
        return None, float('inf')

    # Reconstruct the route
    route = []
    current = destination

    while current is not None:
        route.append(current)
        current = previous[current]

    route.reverse()

    return route, distances[destination]

# **The Sample Game Map**

In [3]:
game_map = {
    "Village": [
        ("Forest Path", 3),
        ("Mountain Path", 7)
    ],

    "Forest Path": [
        ("Village", 3),
        ("Cave", 4)
    ],

    "Mountain Path": [
        ("Village", 7),
        ("Cave", 2)
    ],

    "Cave": [
        ("Forest Path", 4),
        ("Mountain Path", 2)
    ]
}

route, cost = dijkstra(game_map, "Village", "Cave")

print("Recommended route:", route)
print("Total route cost:", cost)

Recommended route: ['Village', 'Forest Path', 'Cave']
Total route cost: 7


# **The Automated tests part**

In [4]:
# Test 1: Find the shortest route
route, cost = dijkstra(game_map, "Village", "Cave")

assert route == ["Village", "Forest Path", "Cave"]
assert cost == 7

print("Test 1 passed: Shortest route")


# Test 2: Starting point and destination are the same
route, cost = dijkstra(game_map, "Village", "Village")

assert route == ["Village"]
assert cost == 0

print("Test 2 passed: Same starting point and destination")


# Test 3: Destination is unreachable
disconnected_map = {
    "Village": [],
    "Cave": []
}

route, cost = dijkstra(disconnected_map, "Village", "Cave")

assert route is None
assert cost == float('inf')

print("Test 3 passed: Unreachable destination")


# Test 4: Invalid destination
try:
    dijkstra(game_map, "Village", "Castle")
    assert False, "Expected a ValueError"
except ValueError:
    print("Test 4 passed: Invalid destination")


# Test 5: Negative route cost
negative_map = {
    "Village": [("Cave", -5)],
    "Cave": []
}

try:
    dijkstra(negative_map, "Village", "Cave")
    assert False, "Expected a ValueError"
except ValueError:
    print("Test 5 passed: Negative route cost")

# Test 6: Empty graph
empty_map = {}

try:
    dijkstra(empty_map, "Village", "Cave")
    print("Test 6 failed: Empty graph was accepted")
except ValueError:
    print("Test 6 passed: Empty graph rejected")


print("\nAll automated tests passed!")

Test 1 passed: Shortest route
Test 2 passed: Same starting point and destination
Test 3 passed: Unreachable destination
Test 4 passed: Invalid destination
Test 5 passed: Negative route cost
Test 6 passed: Empty graph rejected

All automated tests passed!


# **To create larger test maps for benchmarking**

In [5]:
def generate_graph(number_of_nodes, extra_edges=2, seed=42):

    rng = random.Random(seed)

    graph = {
        f"Location_{i}": []
        for i in range(number_of_nodes)
    }

    # Connect every location to the next one
    for i in range(number_of_nodes - 1):
        a = f"Location_{i}"
        b = f"Location_{i + 1}"
        cost = rng.randint(1, 10)

        graph[a].append((b, cost))
        graph[b].append((a, cost))

    # Add extra connections for alternative routes
    for _ in range(extra_edges * number_of_nodes):
        a = rng.randrange(number_of_nodes)
        b = rng.randrange(number_of_nodes)

        if a == b:
            continue

        node_a = f"Location_{a}"
        node_b = f"Location_{b}"
        cost = rng.randint(1, 10)

        graph[node_a].append((node_b, cost))
        graph[node_b].append((node_a, cost))

    return graph


# test check
test_graph = generate_graph(10)

route, cost = dijkstra(
    test_graph,
    "Location_0",
    "Location_9"
)

print("Example route:", route)
print("Example cost:", cost)

Example route: ['Location_0', 'Location_1', 'Location_6', 'Location_9']
Example cost: 5


# **The Automated Benchmark**

In [6]:
def benchmark_dijkstra(sizes=(100, 500, 1000, 2000), repeats=5):

    results = []

    for size in sizes:
        graph = generate_graph(size)

        start = "Location_0"
        destination = f"Location_{size - 1}"

        # Warm-up run and basic correctness check
        route, cost = dijkstra(graph, start, destination)

        assert route is not None
        assert route[0] == start
        assert route[-1] == destination

        # Verify that the returned cost matches the route
        route_cost = 0

        for i in range(len(route) - 1):
            current = route[i]
            next_location = route[i + 1]

            matching_costs = [
                weight
                for neighbor, weight in graph[current]
                if neighbor == next_location
            ]

            assert matching_costs, "Route contains an invalid edge"

            route_cost += min(matching_costs)

        assert route_cost == cost

        # Measure execution time
        times = []

        for _ in range(repeats):
            start_time = time.perf_counter()

            route, cost = dijkstra(graph, start, destination)

            end_time = time.perf_counter()

            times.append(end_time - start_time)

        median_time = statistics.median(times)

        results.append({
            "Locations": size,
            "Connections": sum(len(edges) for edges in graph.values()) // 2,
            "Median Runtime (ms)": round(median_time * 1000, 3),
            "Route Cost": cost
        })

    return results


benchmark_results = benchmark_dijkstra()

for result in benchmark_results:
    print(result)

{'Locations': 100, 'Connections': 297, 'Median Runtime (ms)': 0.276, 'Route Cost': 13}
{'Locations': 500, 'Connections': 1498, 'Median Runtime (ms)': 1.379, 'Route Cost': 17}
{'Locations': 1000, 'Connections': 2999, 'Median Runtime (ms)': 4.132, 'Route Cost': 16}
{'Locations': 2000, 'Connections': 5997, 'Median Runtime (ms)': 6.368, 'Route Cost': 14}


# **Save the benchmark results as csv**

In [7]:
import csv

with open("dijkstra_benchmark_results.csv", "w", newline="") as file:

    writer = csv.DictWriter(
        file,
        fieldnames=benchmark_results[0].keys()
    )

    writer.writeheader()
    writer.writerows(benchmark_results)

print("Benchmark results saved to dijkstra_benchmark_results.csv")

Benchmark results saved to dijkstra_benchmark_results.csv


# **To download the csv**

In [8]:
from google.colab import files

files.download("dijkstra_benchmark_results.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>